In [2]:
# imports
import pandas as pd

In [3]:
# Read csv

df = pd.read_csv('../../data/raw_data/sub/subway_raw.csv')

# Create the combined description
df['Entrance Description'] = (
    df['Station Name'].fillna('') + ' - ' +
    df['North South Street'].fillna('') + ' & ' +
    df['East West Street'].fillna('') + ' (' +
    df['Corner'].fillna('') + ')'
)

# Aggregate unique routes: Collect non-empty Route1-Route11 into a comma separated list
route_columns = [f'Route{i}' for i in range(1, 12)]
df['Routes'] = df[route_columns].apply(lambda row: ','.join([str(r) for r in row if pd.notna(r) and str(r).strip() != '']), axis=1)
# If no routes, set to empty string
df['Routes'] = df['Routes'].replace('', '')

In [4]:
# Clean csv

# Select relevant columns for the project
cleaned_df = df[['Entrance Description', 'Routes', 'Entrance Latitude', 'Entrance Longitude']].copy()

# Convert coords to float (handle any non-numeric)
cleaned_df['Entrance Latitude'] = pd.to_numeric(cleaned_df['Entrance Latitude'], errors='coerce')
cleaned_df['Entrance Longitude'] = pd.to_numeric(cleaned_df['Entrance Longitude'], errors='coerce')

# Drop rows with missing or invalid coords
cleaned_df = cleaned_df.dropna(subset=['Entrance Latitude', 'Entrance Longitude'])

# Filter to NYC bounding box to remove outliers
cleaned_df = cleaned_df[
    (cleaned_df['Entrance Longitude'].between(-74.3, -73.7)) & 
    (cleaned_df['Entrance Latitude'].between(40.5, 40.9))
]

# Remove duplicates based on exact entrance coordinates
cleaned_df = cleaned_df.drop_duplicates(subset=['Entrance Latitude', 'Entrance Longitude'])

# Reset index for clean output
cleaned_df.reset_index(drop=True, inplace=True)


In [5]:
# Save and Preview


cleaned_df.to_csv('../../data/clean_data/sub/cleaned_subway_entrances.csv', index=False)
print("Cleaned shape:", cleaned_df.shape)
print(cleaned_df.head())

Cleaned shape: (1853, 4)
               Entrance Description Routes  Entrance Latitude  \
0  25th St - 4th Ave & 25th St (SW)      R          40.660489   
1  25th St - 4th Ave & 25th St (SE)      R          40.660323   
2  36th St - 4th Ave & 36th St (NW)    N,R          40.654676   
3  36th St - 4th Ave & 36th St (NE)    N,R          40.654365   
4  36th St - 4th Ave & 36th St (NW)    N,R          40.654490   

   Entrance Longitude  
0          -73.998220  
1          -73.997952  
2          -74.004306  
3          -74.004113  
4          -74.004499  
